In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import pandas as pd
import glob

In [4]:
path_to_observations = "../data/detection/outputs/observations/"

In [5]:
path_to_ground_covid = "../data/groudtruth/covidToWorld.csv"
path_to_ground_mpox = "../data/groudtruth/mpoxToWorld.csv"

In [6]:
# Function to convert date format from "year-month-day" to "month-year"
def convert_date_format(df, date_column):
    df[date_column] = pd.to_datetime(df[date_column], format="%Y-%m-%d").dt.strftime("%m-%Y")
    return df


In [7]:
df_covid = pd.read_csv(path_to_ground_covid, index_col=0).reset_index()
df_covid = convert_date_format(df_covid.set_index(df_covid.columns[0]),"date")
df_mpox = pd.read_csv(path_to_ground_mpox, index_col=0).reset_index()
df_mpox = convert_date_format(df_mpox.set_index(df_mpox.columns[0]),"date")

In [8]:
df_covid

,date,Cumulative_cases,ID:END_ID,CUI:START_ID,:TYPE
index,,,,,
0,03-2020,1,wkg:26847706,C5203670,isReported
1,03-2020,1,wkg:26847706,C5203670,isReported
2,03-2020,7,wkg:26847706,C5203670,isReported
3,03-2020,24,wkg:26847706,C5203670,isReported
4,03-2020,91,wkg:26847706,C5203670,isReported
...,...,...,...,...,...
139086,02-2024,349304,wkg:5338938963,C5203670,isReported
139087,02-2024,349304,wkg:6436501440,C5203670,isReported
139090,03-2024,349304,wkg:4860232524,C5203670,isReported


In [9]:
def getObservationDictionary(path_to_observations):
        observation_dict = {}
        for file in glob.glob(path_to_observations + "*.csv"):
            source = file.split("/")[-1].replace("_Observation.csv", "")
            source_prefix = '_'.join(source.split("_")[:2])
            df_observation = pd.read_csv(file)
            # Create a new column 'Observation_Name' without modifying 'System_Name'
            df_observation['Observation_Name'] = source_prefix + '_' + df_observation['System_Name'].astype(
                str).str.replace('"', '', regex=False)
            observation_dict[source] = df_observation
        return observation_dict

In [10]:
dict_obs = getObservationDictionary(path_to_observations)

In [19]:
dict_obs.keys()

dict_keys(['Journal_COVID_SentenceMatching', 'Medical_COVID_SentenceMatching', 'Journal_COVID_DocumentMatching', 'Social_COVID_DocumentMatching', 'Medical_COVID_DocumentMatching', 'Medical_Monkeypox_DocumentMatching', 'Medical_Monkeypox_SentenceMatching', 'Social_Monkeypox_DocumentMatching', 'Journal_COVID_ParagraphMatching', 'Journal_Monkeypox_SentenceMatching', 'Journal_Monkeypox_DocumentMatching', 'Medical_COVID_ParagraphMatching', 'Social_COVID_SentenceMatching', 'Social_Monkeypox_SentenceMatching'])

In [20]:
dict_obs['Journal_COVID_SentenceMatching']

,Unnamed: 0,System_Name,System_ID,average_date,KB_Components,KB_IDs,intensity,Observation_Name
0,0,"""China""",wkg:424313582,19.916667,"['host', 'PTI', 'Genus: Coronavirus', 'SARS', ...","['A18644523', 'A11994316', 'A23448874', 'A2410...","[5, 6, 6, 7, 6, 5, 6, 5, 7, 6, 14, 5, 7, 6, 6,...",Journal_COVID_China
1,1,"""India""",wkg:424314145,19.916667,"['serum', 'family', 'Biological', 'Ebola', 'fa...","['A18560685', 'A18572708', 'C1551397', 'A24183...","[9, 9, 5, 13, 9, 6, 5, 16, 9, 5, 6, 6, 6, 5, 5...",Journal_COVID_India
2,2,"""Singapore""",wkg:531668011,19.916667,"['antibodies', 'antibodies', 'antibodies', 'an...","['A18675045', 'C3495458', 'A18600727', 'A18552...","[5, 5, 5, 5, 5, 5, 6, 5]",Journal_COVID_Singapore
3,3,"""United States""",wkg:424317935,19.916667,"['University', 'University', 'news', 'Universi...","['A2887795', 'A0130305', 'A18580982', 'A107626...","[5, 5, 5, 5]",Journal_COVID_United States


In [35]:
dict_dates = {
    "Journal_COVID" : "11-2019",
    "Medical_COVID" : "12-2019",
    "Social_COVID" : "02-2020",
    "Journal_Monkeypox" : "05-2022",
    "Medical_Monkeypox" : "05-2022",
    "Social_Monkeypox" : "05-2022"
}

In [44]:
import pandas as pd
import math
from sklearn.metrics import f1_score

def convert_float_to_month_year(date_float):
    """
    Convert a float date (e.g., 11.2019) into 'MM-YYYY' format.
    """
    year = int(date_float) # Extract the integer part as year
    month = round((date_float - year) * 12)  # Calculate the month
    return str(f"{month:02d}-20{year}")  # Return in MM-YYYY format

# Function to filter ground truth data based on date
def get_ground_truth(df, date):
    """
    Extract ground truth as a set of ID:END_ID values for the given date.
    """
    filtered_df = df[df['date'] == date]
    return set(filtered_df['ID:END_ID'])

# Function to get predictions based on RE method
def get_predictions(dict_obs, re_method, date_mapping):
    """
    Extract predicted System_ID values for all DataFrames matching a given RE method.
    Only include predictions for the relevant dates in date_mapping.
    """
    predictions = set()
    for key, df in dict_obs.items():
        if re_method in key:
            # Match the date corresponding to the source
            for source, target_date in date_mapping.items():
                if source in key:
                    # Convert average_date from float to MM-YYYY format
                    df['converted_date'] = df['average_date'].apply(convert_float_to_month_year)
                    # Filter by date
                    filtered_df = df[df['converted_date'] == str(target_date)]
                    predictions.update(filtered_df['System_ID'])
    return predictions

# Function to compute F1 scores for all RE methods
def compute_f1_scores(dict_obs, dict_dates, df_covid, df_mpox):
    """
    Compute F1 scores for each RE method.
    """
    re_methods = ["DocumentMatching", "ParagraphMatching", "SentenceMatching"]
    f1_scores = {}

    # Iterate over each RE method
    for re_method in re_methods:
        # Get predictions
        y_pred = get_predictions(dict_obs, re_method, dict_dates)
        
        # Get ground truth for COVID and Monkeypox
        y_true = set()
        for source, date in dict_dates.items():
            if "COVID" in source:
                y_true.update(get_ground_truth(df_covid, date))
            elif "Monkeypox" in source:
                y_true.update(get_ground_truth(df_mpox, date))

        # Convert ground truth and predictions to binary form
        all_ids = y_true.union(y_pred)  # Combined space for comparison
        y_true_binary = [1 if id_ in y_true else 0 for id_ in all_ids]
        y_pred_binary = [1 if id_ in y_pred else 0 for id_ in all_ids]

        # Calculate F1 score
        combined_f1_score = f1_score(y_true_binary, y_pred_binary, average='binary')

        # Save result
        f1_scores[re_method] = combined_f1_score

    return f1_scores

# Example Usage
f1_scores = compute_f1_scores(dict_obs, dict_dates, df_covid, df_mpox)

# Print F1 scores
for method, score in f1_scores.items():
    print(f"F1 Score for {method}: {score:.4f}")


F1 Score for DocumentMatching: 0.2140
F1 Score for ParagraphMatching: 0.0548
F1 Score for SentenceMatching: 0.2157


In [45]:
import pandas as pd
import math
from sklearn.metrics import precision_score

def convert_float_to_month_year(date_float):
    """
    Convert a float date (e.g., 11.2019) into 'MM-YYYY' format.
    """
    year = int(date_float)  # Extract the integer part as year
    month = round((date_float - year) * 12)  # Calculate the month
    return str(f"{month:02d}-20{year}")  # Return in MM-YYYY format

# Function to filter ground truth data based on date
def get_ground_truth(df, date):
    """
    Extract ground truth as a set of ID:END_ID values for the given date.
    """
    filtered_df = df[df['date'] == date]
    return set(filtered_df['ID:END_ID'])

# Function to get predictions based on RE method
def get_predictions(dict_obs, re_method, date_mapping):
    """
    Extract predicted System_ID values for all DataFrames matching a given RE method.
    Only include predictions for the relevant dates in date_mapping.
    """
    predictions = set()
    for key, df in dict_obs.items():
        if re_method in key:
            # Match the date corresponding to the source
            for source, target_date in date_mapping.items():
                if source in key:
                    # Convert average_date from float to MM-YYYY format
                    df['converted_date'] = df['average_date'].apply(convert_float_to_month_year)
                    # Filter by date
                    filtered_df = df[df['converted_date'] == str(target_date)]
                    predictions.update(filtered_df['System_ID'])
    return predictions

# Function to compute Precision scores for all RE methods
def compute_precision_scores(dict_obs, dict_dates, df_covid, df_mpox):
    """
    Compute Precision scores for each RE method.
    """
    re_methods = ["DocumentMatching", "ParagraphMatching", "SentenceMatching"]
    precision_scores = {}

    # Iterate over each RE method
    for re_method in re_methods:
        # Get predictions
        y_pred = get_predictions(dict_obs, re_method, dict_dates)
        
        # Get ground truth for COVID and Monkeypox
        y_true = set()
        for source, date in dict_dates.items():
            if "COVID" in source:
                y_true.update(get_ground_truth(df_covid, date))
            elif "Monkeypox" in source:
                y_true.update(get_ground_truth(df_mpox, date))

        # Convert ground truth and predictions to binary form
        all_ids = y_true.union(y_pred)  # Combined space for comparison
        y_true_binary = [1 if id_ in y_true else 0 for id_ in all_ids]
        y_pred_binary = [1 if id_ in y_pred else 0 for id_ in all_ids]

        # Calculate Precision score
        combined_precision_score = precision_score(y_true_binary, y_pred_binary, zero_division=0)

        # Save result
        precision_scores[re_method] = combined_precision_score

    return precision_scores

# Example Usage
precision_scores = compute_precision_scores(dict_obs, dict_dates, df_covid, df_mpox)

# Print Precision scores
for method, score in precision_scores.items():
    print(f"Precision Score for {method}: {score:.4f}")


Precision Score for DocumentMatching: 0.2148
Precision Score for ParagraphMatching: 0.4000
Precision Score for SentenceMatching: 0.3235


In [46]:
import pandas as pd
import math
from sklearn.metrics import recall_score

def convert_float_to_month_year(date_float):
    """
    Convert a float date (e.g., 11.2019) into 'MM-YYYY' format.
    """
    year = int(date_float)  # Extract the integer part as year
    month = round((date_float - year) * 12)  # Calculate the month
    return str(f"{month:02d}-20{year}")  # Return in MM-YYYY format

# Function to filter ground truth data based on date
def get_ground_truth(df, date):
    """
    Extract ground truth as a set of ID:END_ID values for the given date.
    """
    filtered_df = df[df['date'] == date]
    return set(filtered_df['ID:END_ID'])

# Function to get predictions based on RE method
def get_predictions(dict_obs, re_method, date_mapping):
    """
    Extract predicted System_ID values for all DataFrames matching a given RE method.
    Only include predictions for the relevant dates in date_mapping.
    """
    predictions = set()
    for key, df in dict_obs.items():
        if re_method in key:
            # Match the date corresponding to the source
            for source, target_date in date_mapping.items():
                if source in key:
                    # Convert average_date from float to MM-YYYY format
                    df['converted_date'] = df['average_date'].apply(convert_float_to_month_year)
                    # Filter by date
                    filtered_df = df[df['converted_date'] == str(target_date)]
                    predictions.update(filtered_df['System_ID'])
    return predictions

# Function to compute Recall scores for all RE methods
def compute_recall_scores(dict_obs, dict_dates, df_covid, df_mpox):
    """
    Compute Recall scores for each RE method.
    """
    re_methods = ["DocumentMatching", "ParagraphMatching", "SentenceMatching"]
    recall_scores = {}

    # Iterate over each RE method
    for re_method in re_methods:
        # Get predictions
        y_pred = get_predictions(dict_obs, re_method, dict_dates)
        
        # Get ground truth for COVID and Monkeypox
        y_true = set()
        for source, date in dict_dates.items():
            if "COVID" in source:
                y_true.update(get_ground_truth(df_covid, date))
            elif "Monkeypox" in source:
                y_true.update(get_ground_truth(df_mpox, date))

        # Convert ground truth and predictions to binary form
        all_ids = y_true.union(y_pred)  # Combined space for comparison
        y_true_binary = [1 if id_ in y_true else 0 for id_ in all_ids]
        y_pred_binary = [1 if id_ in y_pred else 0 for id_ in all_ids]

        # Calculate Recall score
        combined_recall_score = recall_score(y_true_binary, y_pred_binary, zero_division=0)

        # Save result
        recall_scores[re_method] = combined_recall_score

    return recall_scores

# Example Usage
recall_scores = compute_recall_scores(dict_obs, dict_dates, df_covid, df_mpox)

# Print Recall scores
for method, score in recall_scores.items():
    print(f"Recall Score for {method}: {score:.4f}")


Recall Score for DocumentMatching: 0.2132
Recall Score for ParagraphMatching: 0.0294
Recall Score for SentenceMatching: 0.1618
